# SigAlg's `SigmaAlgebra` class

In [50]:
# If running in Google Colab, uncomment the line below and run this cell first
# !pip install sigalg

The `SigmaAlgebra` class in SigAlg is the fundamental class for representing $\sigma$-algebras on sample spaces. The API reference is [here](https://johnmyers-phd.com/sigalg/api/core/#sigalg.core.SigmaAlgebra){target="_blank"}.

## Mathematical definition

A *$\sigma$-algebra* $\mathcal{F}$ on a set $\Omega$ is a collection of subsets of $\Omega$ that contains $\Omega$, and is closed under complementation and countable unions. In the case that $\Omega$ is finite (as it always is, in SigAlg), then $\mathcal{F}$ obviously needs only to be closed under finite unions.

A $\sigma$-algebra $\mathcal{F}$ on a finite set $\Omega$ determines its *atoms*, which are the nonempty sets $A\in \mathcal{F}$ that are *minimal* with respect to subset inclusion, in the sense that if $B\in \mathcal{F}$ is nonempty and $B\subset A$, then necessarily $A=B$. And conversely, $\mathcal{F}$ is completely recoverable from its atoms, in the sense that every $B\in \mathcal{F}$ is a union of atoms. The atoms partition the set $\Omega$, which means that the atoms are pairwise disjoint and their union is all of $\Omega$.

If $\{A_i\}_{i\in I}$ is the set of atoms, indexed by a finite set $I$, then there is a mapping $\Omega \to I$ given by $\omega \mapsto i$, where $A_i$ is the unique atom that contains $\omega$. This mapping is what SigAlg uses to represent $\sigma$-algebras. The indices in $I$ are called *atom identifiers*.

In SigAlg, an instance `F` of `SigmaAlgebra` represents such a $\sigma$-algebra $\mathcal{F}$. The instance carries:
- A `sample_space` attribute representing $\Omega$
- A `data` attribute (a `pd.Series`) representing the mapping $\omega \mapsto i$
- Various properties for accessing atoms and atom identifiers

## API examples

### Creating $\sigma$-algebras

#### From dictionaries

Begin by defining a sample space $\Omega = \{0,1,2,3,4\}$.

In [51]:
from sigalg.core import SampleSpace, SigmaAlgebra

Omega = SampleSpace().from_sequence(size=5)

print(Omega)

Sample space 'Omega':
[0, 1, 2, 3, 4]


Create a $\sigma$-algebra $\mathcal{F}$ with atoms $A_0 = \{0,1,2\}$, $A_1 = \{3\}$, and $A_2 = \{4\}$:

In [52]:
F = SigmaAlgebra(sample_space=Omega, name="F").from_dict(
    {
        0: 0,
        1: 0,
        2: 0,
        3: 1,
        4: 2,
    }
)
print(F)

Sigma algebra 'F':
        atom ID
sample         
0             0
1             0
2             0
3             1
4             2


If the sample space is not provided, it will be automatically generated from the dictionary keys:

In [53]:
G = SigmaAlgebra(name="G").from_dict(
    {
        "a": 0,
        "b": 0,
        "c": 1,
    }
)
print(G, "\n")
print(f"Generated sample space: {G.sample_space}")

Sigma algebra 'G':
        atom ID
sample         
a             0
b             0
c             1 

Generated sample space: Sample space 'Omega':
['a', 'b', 'c']


#### From `pd.Series` objects

Create a $\sigma$-algebra from a series:

In [54]:
import pandas as pd

s = pd.Series([0, 0, 1, 1, 2], index=["s0", "s1", "s2", "s3", "s4"])

H = SigmaAlgebra(name="H").from_pandas(s)
print(H)

Sigma algebra 'H':
        atom ID
sample         
s0            0
s1            0
s2            1
s3            1
s4            2


### Factory methods

#### The trivial $\sigma$-algebra

The *trivial $\sigma$-algebra* on a set $\Omega$ consists of only the sets $\Omega$ and $\emptyset$. Its single atom is $\Omega$ itself. It is the coarsest $\sigma$-algebra on $\Omega$.

In [55]:
Omega = SampleSpace().from_sequence(size=4)

trivial = SigmaAlgebra.trivial(sample_space=Omega, name="trivial")
print(trivial)

Sigma algebra 'trivial':
        atom ID
sample         
0             0
1             0
2             0
3             0


#### The power-set $\sigma$-algebra

The *power-set $\sigma$-algebra* on a set $\Omega$ consists of all subsets of $\Omega$. Its atoms are all singleton subsets. It is the finest $\sigma$-algebra on $\Omega$.

In [56]:
power_set = SigmaAlgebra.power_set(sample_space=Omega, name="power_set")
print(power_set)

Sigma algebra 'power_set':
        atom ID
sample         
0             0
1             1
2             2
3             3


#### $\sigma$-algebra generated by a random vector

Given a random vector $X: \Omega \to \mathbb{R}^d$, the *$\sigma$-algebra generated by $X$* is the smallest $\sigma$-algebra $\sigma(X)$ on $\Omega$ with respect to which $X$ is measurable. Its atoms are the nonempty level sets of $X$, i.e., the sets of the form $\{\omega \in \Omega : X(\omega) = x\}$ for $x\in \mathbb{R}^d$.

In [57]:
from sigalg.core import RandomVector

X = RandomVector(domain=Omega, name="X").from_dict(
    {
        0: (1, 2),
        1: (1, 2),
        2: (3, 4),
        3: (5, 6),
    }
)

sigma_X = SigmaAlgebra.from_random_vector(rv=X)
print(sigma_X)

Sigma algebra 'sigma(X)':
       atom ID
sample        
0       (1, 2)
1       (1, 2)
2       (3, 4)
3       (5, 6)


Notice that sample points $0$ and $1$ are in the same atom because they have the same value under $X$.

#### $\sigma$-algebra generated by an event

Given a nonempty event $E \subset \Omega$ which is not equal to $\Omega$ itself, the *$\sigma$-algebra generated by $E$* is the smallest $\sigma$-algebra $\sigma(E)$ on $\Omega$ that contains $E$. It has atoms $E$ and $E^c$ (provided both are nonempty).

In [58]:
E = Omega.get_event([0, 1], name="E")

sigma_E = SigmaAlgebra.from_event(event=E)
print(sigma_E)

Sigma algebra 'sigma(E)':
        atom ID
sample         
0             1
1             1
2             0
3             0


### Properties of $\sigma$-algebras

#### Data and sample space

Access the underlying data and sample space:

In [59]:
F = SigmaAlgebra(name="F").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
        4: 2,
    }
)

print(F.sample_space, "\n")
print(f"Data:\n{F.data}\n")
print(f"Sample ID to atom ID mapping:\n{F.sample_id_to_atom_id}")

Sample space 'Omega':
[0, 1, 2, 3, 4] 

Data:
sample
0    0
1    0
2    1
3    1
4    2
Name: atom ID, dtype: int64

Sample ID to atom ID mapping:
{0: 0, 1: 0, 2: 1, 3: 1, 4: 2}


#### Number of atoms and atom IDs

Access information about atoms:

In [60]:
print(f"Number of atoms: {F.num_atoms}\n")
print(f"Atom IDs: {F.atom_ids}\n")
print(f"Atom ID to sample IDs:\n{F.atom_id_to_sample_ids}\n")
print(f"Atom ID to cardinality:\n{F.atom_id_to_cardinality}")

Number of atoms: 3

Atom IDs: [np.int64(0), np.int64(1), np.int64(2)]

Atom ID to sample IDs:
{0: [0, 1], 1: [2, 3], 2: [4]}

Atom ID to cardinality:
{0: 2, 1: 2, 2: 1}


#### Atoms as events

Get atoms as `Event` objects:

In [61]:
print("Atom ID to event:\n")
for atom_id, event in F.atom_id_to_event.items():
    print(f"Atom {atom_id}:\n{event}\n")

Atom ID to event:

Atom 0:
Event '0':
[0, 1]

Atom 1:
Event '1':
[2, 3]

Atom 2:
Event '2':
[4]



Get a list of all atoms:

In [62]:
atoms = F.to_atoms()
for atom in atoms:
    print(atom)

Event '0':
[0, 1]
Event '1':
[2, 3]
Event '2':
[4]


### Methods

#### Getting the atom containing a sample point

For a given sample point, find which atom contains it:

In [63]:
atom = F.get_atom_containing(sample_id=1)
print(atom)

Event 'A':
[0, 1]


#### Checking if an event is measurable

An event $E$ is *$\mathcal{F}$-measurable* if it is a union of atoms of $\mathcal{F}$. Check measurability:

In [64]:
Omega = SampleSpace().from_sequence(size=5)
F = SigmaAlgebra(sample_space=Omega, name="F").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
        4: 2,
    }
)

# This event is a union of atoms {0,1} and {2,3}
E1 = Omega.get_event([0, 1, 2, 3], name="E1")
print(f"Is E1 F-measurable? {F.is_measurable(E1)}\n")

# This event is not a union of atoms
E2 = Omega.get_event([0, 2, 4], name="E2")
print(f"Is E2 F-measurable? {F.is_measurable(E2)}")

Is E1 F-measurable? True

Is E2 F-measurable? False


#### Checking containment

Check if an event belongs to a sigma-algebra (same as checking measurability):

In [65]:
print(f"Is E1 in F? {E1 in F}\n")
print(f"Is E2 in F? {E2 in F}")

Is E1 in F? True

Is E2 in F? False


### Operations and comparisons between $\sigma$-algebras

#### Join of $\sigma$-algebras

Given two $\sigma$-algebras $\mathcal{F}$ and $\mathcal{G}$, their *join* $\mathcal{F} \vee \mathcal{G}$ is the smallest $\sigma$-algebra that contains both $\mathcal{F}$ and $\mathcal{G}$. Its atoms are the nonempty intersections of atoms of $\mathcal{F}$ with atoms of $\mathcal{G}$. Thus, the atom identifiers of the join may be taken as ordered pairs of atom identifiers of $\mathcal{F}$ and $\mathcal{G}$, with the understanding that the pair $(i,j)$ corresponds to the intersection of the $i$-th atom of $\mathcal{F}$ with the $j$-th atom of $\mathcal{G}$.

In [66]:
Omega = SampleSpace().from_sequence(size=4)

F = SigmaAlgebra(sample_space=Omega, name="F").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
    }
)

G = SigmaAlgebra(sample_space=Omega, name="G").from_dict(
    {
        0: 0,
        1: 1,
        2: 0,
        3: 1,
    }
)

print(F, "\n")
print(G, "\n")
print(F | G)

Sigma algebra 'F':
        atom ID
sample         
0             0
1             0
2             1
3             1 

Sigma algebra 'G':
        atom ID
sample         
0             0
1             1
2             0
3             1 

Sigma algebra 'join':
       atom ID
sample        
0       (0, 0)
1       (0, 1)
2       (1, 0)
3       (1, 1)


#### Sub-$\sigma$-algebras

A $\sigma$-algebra $\mathcal{F}$ is a *sub-$\sigma$-algebra* of another $\sigma$-algebra $\mathcal{G}$ if $\mathcal{F} \subset \mathcal{G}$ as sets. Equivalently, every atom of $\mathcal{F}$ is a union of atoms of $\mathcal{G}$. In this case, we also say that $\mathcal{F}$ is *coarser* than $\mathcal{G}$, and that $\mathcal{G}$ is *finer* than $\mathcal{F}$.

In [67]:
Omega = SampleSpace().from_sequence(size=4)

trivial = SigmaAlgebra.trivial(sample_space=Omega, name="trivial")
middle = SigmaAlgebra(sample_space=Omega, name="middle").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
    }
)
power_set = SigmaAlgebra.power_set(sample_space=Omega, name="power_set")

print(trivial, "\n")
print(middle, "\n")
print(power_set, "\n")

print(f"Is trivial ⊆ middle? {trivial <= middle}")
print(f"Is trivial < middle? {trivial < middle}")
print(f"Is middle ⊆ power_set? {middle <= power_set}")
print(f"Is middle < power_set? {middle < power_set}")
print(f"Is trivial ⊆ power_set? {trivial <= power_set}")
print(f"Is power_set ⊆ trivial? {power_set <= trivial}")
print(f"Is middle ⊆ trivial? {middle <= trivial}")

Sigma algebra 'trivial':
        atom ID
sample         
0             0
1             0
2             0
3             0 

Sigma algebra 'middle':
        atom ID
sample         
0             0
1             0
2             1
3             1 

Sigma algebra 'power_set':
        atom ID
sample         
0             0
1             1
2             2
3             3 

Is trivial ⊆ middle? True
Is trivial < middle? True
Is middle ⊆ power_set? True
Is middle < power_set? True
Is trivial ⊆ power_set? True
Is power_set ⊆ trivial? False
Is middle ⊆ trivial? False


#### Equality

Two $\sigma$-algebras are equal if they are equal as sets. Equivalently, two $\sigma$-algebras are equal if they have the same atoms.

In [68]:
F1 = SigmaAlgebra(name="F1").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
    }
)

F2 = SigmaAlgebra(name="F2").from_dict(
    {
        0: 5,
        1: 5,
        2: 7,
    }
)

print(f"F1 == F2? {F1 == F2}")

F1 == F2? True


Note that the atom IDs themselves don't matter, only the partition structure.